# NeuroPAL whole-brain activity (Yemini et al. 2021) from the paper's Zenodo record

**Paper.** Yemini et al. (2021), *Cell* 184:272-288,
[doi:10.1016/j.cell.2020.12.012](https://doi.org/10.1016/j.cell.2020.12.012); claims written from
the accepted author manuscript (PMC10494711) with Methods S1 and S2.

**Data.** These claims use data the dandiset (DANDI:000715, version 0.241009.1514) does not hold:
the whole-brain calcium activity of 21 heads and 21 tails of strain OH16230, which the paper
deposited on Zenodo, [doi:10.5281/zenodo.3906530](https://doi.org/10.5281/zenodo.3906530) (one
published version; files `Head_Activity_OH16230.mat` and `Tail_Activity_OH16230.mat` in
`NeuroPAL data.zip`). They are `data: elsewhere` claims and are counted apart from the dandiset.
The dandiset's own claims are in `reproduce_yemini_2021.ipynb`.

| Claim | Status here |
|---|---|
| fig6d-ase-nacl-asymmetry | tested |
| fig6ef-nacl-novel-interneurons | tested |
| fig6de-asj-ric-stimulus-specific | tested |
| tables6-tail-pvq-and-post-stimulus | tested |
| figs1h-avf-cyclic-activity | tested |
| other whole-brain claims (6C, 6D AWC, 6F, 6G-I, 6J-K) | not tested; see the record |

In [ ]:
import time
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

import wholebrain as wb
import neuropal as npl   # names the dandiset of this entry; its data are not used here

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
t0 = time.time()
raw = {part: wb.load(part) for part in ("head", "tail")}
print(f"read the head and tail files from Zenodo {wb.ZENODO_DOI} in {time.time() - t0:.0f} s "
      f"(dandiset of this entry: {npl.DANDISET_ID} {npl.DANDISET_VERSION}, not used here)")

## Data check

In [ ]:
D = {part: wb.tidy(m) for part, m in raw.items()}
for part, d in D.items():
    lens = [len(x) for x in d["traces"].values()]
    vals = np.concatenate([x[~np.isnan(x)] for x in d["traces"].values()])
    print(f"{part}: {len(d['files'])} files, {len({k[0] for k in d['traces']})} neuron classes, {len(d['traces'])} traces, "
          f"fps {d['fps']}, trace length {min(lens)} to {max(lens)} samples ({min(lens) / d['fps']:.0f} to {max(lens) / d['fps']:.0f} s), "
          f"normalised values {vals.min():.2f} to {vals.max():.2f}, NaN samples per trace (median) "
          f"{int(np.median([np.isnan(x).sum() for x in d['traces'].values()]))}")
    print(f"  stimuli {d['stim_names']}, windows {d['stim_times'].tolist()} s, orders per file:",
          pd.Series([tuple(r) for r in d["stims"]]).value_counts().to_dict())
print("\nPaper: 21 heads (189 neurons) and 21 tails (42 neurons), 4 min at about 4 Hz, 10 s pulses one minute apart, in rotated order.")

In [ ]:
# The order of stimuli: each row of `stims` is a permutation that is its own inverse, so the two
# readings of the README ("use these indices with both stim_names and stim_times") give the same
# windows. The labels are checked against the published responses the paper used to select the
# worms: ASE excitation to NaCl and AWC inhibition to odors, and against windows shifted in time.
H = D["head"]
print("Every order is its own inverse:", all((np.argsort(r - 1) == r - 1).all() for r in H["stims"]))
rows = []
for stim, cls, sign in (("nacl", "ASE", 1), ("butanone", "AWC", -1), ("pentanedione", "AWC", -1)):
    for shift in (0, -30, 30):
        Hs = dict(H, stim_times=H["stim_times"] + shift)
        ch = {}
        for k in H["traces"]:
            if k[0] == cls:
                pre, dur, _ = wb.window_means(Hs, k, stim)
                ch.setdefault(k[2], []).append(dur - pre)
        worms = sum(any(sign * c > 0 for c in v) for v in ch.values())
        allc = np.concatenate(list(ch.values()))
        rows.append(dict(stimulus=stim, neuron=cls, shift_s=shift, worms_with_published_sign=worms, of=len(ch),
                         mean_change=allc.mean(), paired_t_p=stats.ttest_1samp(allc, 0).pvalue))
print(pd.DataFrame(rows).round(4).to_string(index=False))

With the stated windows ASE rises to NaCl and AWC falls to both odors in nearly every worm, and
the changes shrink or vanish when the windows are moved 30 s, so the stimulus labels and times are
consistent with the responses.

The STAR Methods remove neurons within 5 um whose traces correlate at 0.95 or more; the README
says the files keep them. Three readings are run: no removal, removal of both members of each
pair (the literal reading of "Neurons within 5 um distance of each other, whose traces showed at
least 95% correlation, were deemed too similar and removed"), and removal of one member. The paper
supports none over the others. Removing both drops about a fifth of the traces (next cell), so the
readings can differ; a claim whose outcome differs between them is blocked.

In [ ]:
import copy
readings = {}
for label, kw in (("none", None), ("both removed", dict(both=True)), ("one removed", dict(both=False))):
    readings[label] = {}
    for part in ("head", "tail"):
        d = copy.deepcopy(D[part])
        n_removed = len(wb.remove_similar(d, **kw)) if kw else 0
        readings[label][part] = d
        print(f"{label:13s} {part}: {n_removed} traces removed, {len(d['traces'])} kept")
res = {lab: {part: {s: pd.DataFrame(wb.responses(d, s)).set_index("neuron") for s in d["stim_names"]}
             for part, d in r.items()} for lab, r in readings.items()}
for lab in res:
    print(lab, "significant responses (q <= 0.05), head:", {s: int((t.q <= 0.05).sum()) for s, t in res[lab]["head"].items()},
          "tail:", {s: int((t.q <= 0.05).sum()) for s, t in res[lab]["tail"].items()})

## fig6d-ase-nacl-asymmetry

> NaCl (160 mM) evokes left/right asymmetric responses in the ASE neurons, with ASER increasing
> its calcium.

Criterion: two-tailed unpaired t test between left and right responses, p of at most 0.05.

In [ ]:
rows = []
for lab, r in readings.items():
    d = r["head"]
    ch = {side: np.array([np.subtract(*wb.window_means(d, k, "nacl")[1::-1]) for k in d["traces"] if k[0] == "ASE" and k[1] == side])
          for side in ("L", "R")}
    t = stats.ttest_ind(ch["L"], ch["R"])
    rows.append(dict(reading=lab, n_L=len(ch["L"]), n_R=len(ch["R"]), mean_change_ASEL=ch["L"].mean(), mean_change_ASER=ch["R"].mean(),
                     ASER_up_worms=int((ch["R"] > 0).sum()), t=t.statistic, p=t.pvalue,
                     ASER_vs_0_p=stats.ttest_1samp(ch["R"], 0).pvalue))
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** ASEL rises by 1.01 and ASER by 0.42 (normalised units) on average over
21 worms each, a left/right difference with t = 3.82, p = 0.00046; ASER rises in 14 of 21 worms
(mean against 0, p = 0.0032). With the similar neurons removed (both members) p = 0.0019 and
ASER still rises (0.46).

## fig6ef-nacl-novel-interneurons

> NaCl evokes significant responses in six interneurons with no previously known function, AIN,
> AVF, AVH, AVJ, I2 and MI.

In [ ]:
six = ["AIN", "AVF", "AVH", "AVJ", "I2", "MI"]
for lab in res:
    t = res[lab]["head"]["nacl"].loc[six, ["n", "mean_change", "p", "q"]]
    print(f"{lab}: all six significant (q <= 0.05): {bool((t.q <= 0.05).all())}")
    print(t.to_string(float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** All six neurons respond to NaCl with q at most 0.05 under the three
readings (largest q 0.041 for AVF without removal, 0.039 for AVJ with both members removed), and
all six rise.

## fig6de-asj-ric-stimulus-specific

> The sets of responding neurons overlap but differ by stimulus; ASJ is excited by NaCl and
> inhibited by both odors, and RIC is excited only by 2,3-pentanedione.

In [ ]:
for lab in res:
    rows = []
    for n in ("ASJ", "RIC"):
        for s in ("nacl", "butanone", "pentanedione"):
            t = res[lab]["head"][s].loc[n]
            rows.append(dict(neuron=n, stimulus=s, n=int(t.n), mean_change=t.mean_change, q=t.q, significant=bool(t.q <= 0.05)))
    tab = pd.DataFrame(rows)
    asj = tab[(tab.neuron == "ASJ")].set_index("stimulus")
    ric = tab[(tab.neuron == "RIC")].set_index("stimulus")
    asj_ok = asj.loc["nacl"].significant and asj.loc["nacl"].mean_change > 0 and all(asj.loc[o].significant and asj.loc[o].mean_change < 0 for o in ("butanone", "pentanedione"))
    ric_ok = ric.loc["pentanedione"].significant and ric.loc["pentanedione"].mean_change > 0 and not ric.loc["nacl"].significant and not ric.loc["butanone"].significant
    sets = {s: set(res[lab]["head"][s].index[res[lab]["head"][s].q <= 0.05]) for s in ("nacl", "butanone", "pentanedione")}
    print(f"{lab}: ASJ as stated {asj_ok}; RIC as stated {ric_ok}; responding sets {[len(v) for v in sets.values()]}, "
          f"in all three {len(set.intersection(*sets.values()))}, union {len(set.union(*sets.values()))}")
    print(tab.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: blocked: the outcome depends on a detail the paper leaves open.** Without removal of
similar neurons, and with one member of each pair removed, both statements hold: ASJ is excited by
NaCl and inhibited by both odors (q 0.019 and 0.033 for 2,3-pentanedione), and RIC responds only to
2,3-pentanedione (q 0.0044 and 0.019; NaCl q 0.065 and 0.054). With both members removed, ASJ's fall
with 2,3-pentanedione is not significant (q 0.063) and RIC also responds to NaCl (q 0.044). The
responding sets overlap and differ under every reading (9 or 10 neurons respond to all three
stimuli out of 60 to 65).

## tables6-tail-pvq-and-post-stimulus

> In the tail, PVQ is significantly excited by 2,3-pentanedione, and sensory and interneurons show
> significant post-stimulus responses to all three stimuli.

In [ ]:
for lab in res:
    t = res[lab]["tail"]
    pvq = t["pentanedione"].loc["PVQ"]
    print(f"{lab}: PVQ to pentanedione n {int(pvq.n)}, mean change {pvq.mean_change:.4f}, q {pvq.q:.4g}; to butanone q "
          f"{t['butanone'].loc['PVQ'].q:.3g}, to NaCl q {t['nacl'].loc['PVQ'].q:.3g}")
    for s in ("nacl", "butanone", "pentanedione"):
        post = t[s][(t[s].q_post <= 0.05)]
        print(f"   post-stimulus responders to {s}: " + ", ".join(f"{n} (q {v:.3f})" for n, v in post.q_post.items()))

**Result: partially reproduced.** PVQ is excited by 2,3-pentanedione under every reading (mean
change 0.048 to 0.049, q 0.0006 to 0.0008) and not by the other two stimuli. Post-stimulus
responders exist for all three stimuli without removal (NaCl LUA, PHB and PVW) and with one member
removed (NaCl LUA only), but with both members removed none remains for NaCl. So the claim holds at
least in part under every reading, and in full on the criterion under two of the three. Which responders are sensory neurons and
which interneurons is not tested: the record holds no neuron types and the claims file gives none.

## figs1h-avf-cyclic-activity

> In many animals AVF shows robust cyclical activity of about 0.3 Hz throughout the recording,
> regardless of stimuli.

Figure S1H counts peaks per 4-minute interval in traces median filtered at 0.5 s (3 samples, 0.75 s, the shortest odd window at 4 Hz), from five worms
it does not name. The peak rule is not given; it is run at three prominences, and the frequency of
the largest spectral power is shown as a check. All AVF traces are used.

In [ ]:
d = D["head"]
keys = [k for k in d["traces"] if k[0] == "AVF"]
rows = []
for k in keys:
    x = d["traces"][k]
    rows.append(dict(trace=f"{k[1]}{d['files'][k[2]]}", **{f"peaks_per_s_prom{p}": wb.peak_rate(x, d["fps"], p) for p in (0.5, 1.0, 1.5)},
                     spectral_peak_hz=wb.spectral_peak(x, d["fps"])))
avf = pd.DataFrame(rows)
print(f"{len(avf)} AVF traces from {len({k[2] for k in keys})} worms")
print(avf.describe().loc[["mean", "25%", "50%", "75%", "min", "max"]].round(3).to_string())
for c in avf.columns[1:]:
    print(f"{c}: traces within 0.27 to 0.33 Hz {int(avf[c].between(0.27, 0.33).sum())} of {len(avf)}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.5))
k = keys[0]
x = d["traces"][k]
t = np.arange(len(x)) / d["fps"]
ax.plot(t, x, lw=0.7, color="0.2")
for s, e in d["stim_times"]:
    ax.axvspan(s, e, color="C1", alpha=0.3)
ax.set_xlabel("time (s)"); ax.set_ylabel("AVF, normalised")
plt.tight_layout(); plt.show()
print(f"trace {k}: range {np.nanmin(x):.2f} to {np.nanmax(x):.2f}; no point outside the axes")

**Result: not reproduced, under an assumed peak rule.** Over 31 AVF traces from 18 worms the
median peak rate is 0.20, 0.11 and 0.07 per second at prominences of 0.5, 1 and 1.5 standard
deviations, and the median frequency of the largest spectral power is 0.19 Hz; one trace (peak
count) and three traces (spectrum) of 31 fall in 0.27 to 0.33 Hz. The paper's five worms are not
named, so a subset at 0.3 Hz cannot be excluded; this analysis pools all 31 traces from 18 worms.

## Summary

| Claim | Outcome |
|---|---|
| fig6d-ase-nacl-asymmetry | reproduced |
| fig6ef-nacl-novel-interneurons | reproduced |
| fig6de-asj-ric-stimulus-specific | blocked: holds without removal or with one member removed, fails with both removed |
| tables6-tail-pvq-and-post-stimulus | partially reproduced (in full on the criterion under two of the three removal readings) |
| figs1h-avf-cyclic-activity | not reproduced (rests on an assumed peak rule) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

Differences from the paper's method: the removal of similar neurons (5 um, r of 0.95 or more) is
run three ways because the text does not say whether both members of a pair were dropped
(removing both drops 647 of 3,550 head and 161 of 797 tail traces); the
post-stimulus window is taken as 10 s; the peak rule of Figure S1H is not given. About the data:
the record gives no neuron types and no list of the neurons the paper removed, and the files keep
the similar neurons the paper excluded.